# LiDAR Patch Processing

This notebook converts a georeferenced LiDAR mosaic into fixed 256 x 256 training patches. Each output GeoTIFF contains band 1 as the LiDAR surface/residual and band 2 as a validity mask. Patch IDs and georeferencing are preserved so Sentinel-1 windows can be matched later.

The patch grid uses a configurable overlap. Existing Tessa patches can be used instead; in that case, skip the extraction cell and point the later notebooks at the existing patch directory.

In [ ]:
from pathlib import Path
import json
import numpy as np
import rasterio
from rasterio.windows import Window
from rasterio.windows import transform as window_transform

## Configuration

Set `MOSAIC_PATH` to the LiDAR mosaic and choose a region. The target patch size is 256 pixels, matching Tessa's model input.

In [ ]:
REPO_DIR = Path('/Users/jessica/Desktop/project/Michel/RoughNet')
MOSAIC_PATH = Path('/Users/jessica/Desktop/project/Project_code/lidar_data_2024/TukApr16')
REGION = 'tuk'
OUT_DIR = REPO_DIR / 'input_data' / f'lidar_patches_{REGION}'
PATCH_SIZE = 256
OVERLAP = 0.5
MIN_VALID_FRACTION = 0.02
OUT_DIR.mkdir(parents=True, exist_ok=True)

## Locate the mosaic

This accepts either a direct GeoTIFF path or a directory containing one. The source must have a CRS and affine transform.

In [ ]:
def resolve_mosaic(path):
    if path.is_file():
        return path
    candidates = sorted(path.glob('*.tif')) + sorted(path.glob('*.tiff'))
    if not candidates:
        raise FileNotFoundError(f'No GeoTIFF found at {path}')
    return candidates[0]

MOSAIC_PATH = resolve_mosaic(MOSAIC_PATH)
with rasterio.open(MOSAIC_PATH) as src:
    print('Mosaic:', MOSAIC_PATH)
    print('Shape:', src.height, src.width, 'bands:', src.count, 'CRS:', src.crs)
    assert src.crs is not None, 'The LiDAR mosaic needs a CRS for Sentinel-1 matching.'

## Extract patches

Only windows with enough finite LiDAR data are written. The mask is finite where the source has usable data, so downstream training can ignore nodata pixels.

In [ ]:
def extract_lidar_patches(mosaic_path, output_dir, patch_size=256, overlap=0.5, min_valid_fraction=0.02):
    step = max(1, int(round(patch_size * (1.0 - overlap))))
    written = 0
    with rasterio.open(mosaic_path) as src:
        profile = src.profile.copy()
        data_band = src.read(1)
        source_mask = src.read_masks(1) > 0 if src.count == 1 else src.read(2) > 0
        for row in range(0, max(1, src.height - patch_size + 1), step):
            for col in range(0, max(1, src.width - patch_size + 1), step):
                if row + patch_size > src.height or col + patch_size > src.width:
                    continue
                window = Window(col, row, patch_size, patch_size)
                data = data_band[row:row + patch_size, col:col + patch_size].astype(np.float32)
                mask = source_mask[row:row + patch_size, col:col + patch_size].astype(np.float32)
                mask &= np.isfinite(data)
                if float(mask.mean()) < min_valid_fraction:
                    continue
                data = np.where(mask > 0, data, np.nan).astype(np.float32)
                profile.update(driver='GTiff', dtype='float32', count=2, height=patch_size, width=patch_size, transform=window_transform(window, src.transform), nodata=np.nan, compress='deflate')
                output_path = output_dir / f'lidar_patch_{written:05d}.tif'
                with rasterio.open(output_path, 'w', **profile) as dst:
                    dst.write(data, 1)
                    dst.write(mask, 2)
                written += 1
    return written

n_written = extract_lidar_patches(MOSAIC_PATH, OUT_DIR, PATCH_SIZE, OVERLAP, MIN_VALID_FRACTION)
print('Patches written:', n_written)

## Verify the patch contract

Every file should be 256 x 256 with two bands. The CRS and transform are required by the Sentinel-1 collocation notebook.

In [ ]:
patches = sorted(OUT_DIR.glob('lidar_patch_*.tif'))
assert patches, 'No LiDAR patches were produced.'
with rasterio.open(patches[0]) as src:
    assert src.count == 2 and src.shape == (PATCH_SIZE, PATCH_SIZE)
    print('Verified:', patches[0].name, src.shape, src.count, src.crs)